The five generated features (additional headline, keywords, keywords-constrained headline, academic-style headline, and summary) are non-deterministic because the temperature is set to 1. The result can be seen on the "GPT Features (NonStable)".

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os
from openai import OpenAI
import csv

_ = load_dotenv(find_dotenv())
client = OpenAI(
    api_key=os.environ.get('OPENAI_API_KEY')
)

def getContent(inputFile):
    getFile = [f for f in os.listdir(inputFile) if f.endswith('.txt')]
    contents = {}
    for file in getFile:
        try:
            with open(os.path.join(inputFile, file), 'r', encoding='utf-8', errors='replace') as f:
                lines = f.readlines()
                title = lines[0].replace("Title: ", "").strip()
                date = lines[1].replace("Date: ", "").strip()
                contentIndexStart = lines.index("Content:\n") + 1
                content = ''.join(lines[contentIndexStart:])
                contents[file] = (title, date, content.strip())
        except Exception as e:
            print(f"Error reading file {file}: {e}")
    return contents

def getResponses(content, promptlst):
   
    prompts = "\n".join([f"Prompt {i+1}: {prompt}" for i, prompt in enumerate(promptlst)])
    completion = client.chat.completions.create(
         model="gpt-4o",
         messages=[
            {"role": "system", "content": "You are a helpful economist and journalist"},
            {"role": "user", "content": f"{prompts}\n\n{content}\n\n'Please just write the response; you do not need to write prompt 1, etc.'\n\n'Each response only needs to be divided by one newline'"}
        ],
        temperature= 1,
        max_tokens= 1096
    )
    responses = completion.choices[0].message.content.split("\n\n")
    usedToken = completion.usage.total_tokens
    return responses, usedToken

def process(inputFile, outputFile):
    contents = getContent(inputFile)
    rows = []
    promptlst = [
        "Please identify five keywords for the following news article (when generating the keywords, please use the format: ..., ..., ..., ...)",
        "Please come up with a high-quality (attractive, clear, accurate, and inoffensive) headline for the following news article using at least three of the identified keywords, please only generate the headline without quotation marks",
        "Please come up with a high-quality (attractive, clear, accurate, and inoffensive) headline for the following news article, please only generate the headline without quotation marks",
        "Generate an academic-style news headline for the following news article, please only generate the headline without quotation marks",
        "Generate a short summary of the following news article in as few words as possible, please ensure the summary is in one line without creating a newline for a new sentence"
    ]

    for file, (title, date, content) in contents.items():
        responses, usedToken = getResponses(content, promptlst)
        rows.append([date, title] + responses + [usedToken])

    with open(outputFile, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Date", "Original Headline", "Keywords", "Keyword-Constrained Headline", "Generated Headline", "Academic-style Headline", "Generated Summary", "Tokens Used"])
        writer.writerows(rows)

inputFile = "decoy"
outputFile = "decoy"
process(inputFile, outputFile)